In [13]:
import numpy as np, json, requests
OLLAMA = "http://10.42.0.247:11434/"
#GEN_MODEL = "qwen2.5:14b-instruct" # runs on cloudwright after Tuesday
#GEN_MODEL = "qwen3.5:9b"
GEN_MODEL = "ornith:9b"
def ask(query):
    prompt = f"""{query}"""

    r = requests.post(f"{OLLAMA}/api/generate",
        json={"model": GEN_MODEL, "prompt": prompt, "stream": False}, timeout=1000)
        
    resp = r.json()
    print(resp.keys())     
    if 'thinking' in resp:
        print("thinking: ",resp['thinking'])
    
    return resp["response"].strip()


In [38]:
tools_description = """

**Command Interface**
you and i will communicate in json blocks

you will receive a json block like:

{
 "format" : "json_plain",
 "query" : "OPTIONAL query from the user, if this is present, address it",
 "tool_call_result" : "OPTIONAL result of the last tool call, if a tool was called it will start with one line that holds the issued tool call text, followed by the results or error message",
 "thread_summary" : "REQUIRED current summary of this working session"
}

return only a JSON block like the following, no preamble or conclusion

{
 "response" : "OPTIONAL: text to the user, this part will not be executed, but the user will have to reply with a new query if this is present, if you dont need input from the user on this loop omit this key entirely",
 "tool_call" : "OPTIONAL a literal tool call from the exposed list:
    * `ls <options>`: List files in the local working area
    * `cat <filename>`: Display the contents of a file
    * `mkdir <directory name>` : make subdir in the working directory
    * `echo <some content>` : use to create files like echo \"some-content\" > filename_in_working_dir_structure 
    (NOTE you must escape double quotes with backslash \\ if you use them)
    this part will execute, you will receive the results on the next call, only output one tool call as an unformatted string exactly as it would be entered on the commandline",
 "thread_summary" : "REQUIRED a progressive summary of this conversation that includes the information from the thread_summary you were sent, try to keep it under 20k"
}

**Using the Tool**

* the working dir WILL NOT change from turn to turn, issue all commands from the inital working dir
* Issue a command using the following syntax: `command arguments...`
* Use quotes to enclose arguments with spaces (e.g., `cat "example file.txt"`)
* if the result of a tool call is unexpected, consult the user

**Example tool_call Input**

* `ls`
* `ls <some options>`
* `cat example.txt`

Please respond with a json_plain response the starts with '{' and ends with '}', no formatting or unrequired newlines, 
without any additional text or narrative explanation.

make sure to ALWAYS include context in the thread_summary field so that the agent that takes the next action has some

"""


In [39]:
import json
def distillation_prompt(thread_summary):
    """Run distillation with LLM"""
    return f"""You are Model A in a distillation pipeline. Your job is to extract candidate artifacts and determine keep/discard assessments. A second Model B will judge and finalize them.

## What You Do

Convert conversation artifacts into candidate memory artifacts with explicit keep/discard decisions.

## Critical Rule: Decision Reconstruction, Not Style Matching

You must extract ACTUAL claims from the delivered context and produce artifacts that encode those decisions with falsification hooks. If you're just reformatting existing ideas without adding decision content, indicate verdict = DISCARD on the artifact

## Worked Example (follow this pattern)

Input: "The three-layer split (knowledge / cognitive interface / execution) is the strongest structural idea."
→ Extract claim: "PCI's architecture uses a three-layer split"
→ Produce artifact: principles/pci-architecture-layers.md with boundaries and falsification test
→ Decision: KEEP (real file with content)

## Anti-Patterns to Avoid

- Escalation without resistance (always more framework)
- Answering counterarguments with "another schema" instead of an artifact
- Novelty claims without operational distinction
- Governance metadata without specifying actual decision procedure

## Output Format (JSON ONLY — no markdown, no prose)

Return a JSON array of artifacts similar to the following:

{json.dumps(
[
  {
    "name": "principles/pci-distillation-operator",
    "content": "Defines PCI distillation operationally",
    "verdict": "KEEP",
    "reasoning": "Produces real transformation with falsification hook"
  },
  {
    "name": "operators/distillation-run-experiment",
    "content": "Records a held-out evaluation method",
    "verdict": "KEEP",
    "reasoning": "Specific enough to run next week, concrete metrics"
  },
  {
    "name": "governance/decision-procedure",
    "content": "Specifies conflict resolution",
    "verdict": "HOLD",
    "reasoning": "Needs current conflict policy before committing"
  }
]
)}
## Procedure

1. Extract keepable claims (explicit, testable, small)
2. Identify failure modes (anti-patterns)
3. Generate candidate artifacts (any number that make sense)
4. Decide keep/discard for each (sealed verdicts, no "later")

## Falsification

If evaluation shows no decision reconstruction → PCI shrinks to style-transfer. We are aiming to enable continued collaboration.

context to distill follows:
==========================

{thread_summary}

"""
    

In [40]:
def run_distillation(thread):
    distillation_results = ask(distillation_prompt(thread["thread_summary"]))
    print(json.loads(distillation_results))

In [41]:
import json
import subprocess
import pathlib

def execute_command(command):
    process = subprocess.run(command, cwd="./output", shell=True, capture_output=True)
    return process.stdout.decode('utf-8')

# Main loop
def execute(command):
    if command.startswith('ls'):
        output = execute_command(command)
    elif command.startswith('cat'):
        output = execute_command(command)
    elif command.startswith('mkdir'):
        output = execute_command(command)
    elif command.startswith('echo'):
        output = execute_command(command)
    else:
        output = 'Unknown command'
        return output, "error"
    return output, None


In [42]:
from IPython.display import clear_output


start_loop = f"""

hi there, im doing some experiments giving you control over the command line, please dont hurt me, im actually not that bad of a person

the first thing we will do is see if you can use the following tools to create a "projects" directory in the working dir

then create a file there named README.md that contains a hello world statement

if you have questions, ask them per the following interface

{tools_description}
"""

thread = {
 "format" : "json_plain",
 "query" : start_loop,
 "thread_summary" : "thread is starting"
}

for i in range(5):
    print("step", i)
    #clear_output()
    #print(thread)
    reply = ask(thread)
    query = ""
    try:
        reply = json.loads(reply)
        print("thread_summary=", reply.get("thread_summary"))
        next_call = {"format" : "json_plain", "thread_summary" : reply.get("thread_summary", thread["thread_summary"])}

        if 'response' in reply and len(reply['response']) > 0:
            print(GEN_MODEL, reply['response'])
            print("wants to run: ", reply['tool_call'])
            query = input('LLM: $ ')
            
        if 'tool_call' in reply:
            print("tool_call", reply['tool_call'])
            output, err = execute(reply['tool_call'])
            print("output=",output)
            if err:
                print(err)
            next_call["tool_call_result"] = f"""
{reply['tool_call']}
{output}
{f"ERROR: {err}" if err is not None else ""}"""
        
        next_call["query"] = f""" 
{query}
{tools_description}"""
        
        thread = next_call
    except Exception as e:
        print("EXCEPTION:", e)
        break

    

step 0
dict_keys(['model', 'created_at', 'response', 'thinking', 'done', 'done_reason', 'context', 'total_duration', 'load_duration', 'prompt_eval_count', 'prompt_eval_duration', 'eval_count', 'eval_duration'])
thinking:  The user wants me to:
1. Create a "projects" directory in the working dir
2. Create a file there named README.md with "hello world" content

Let me do this with the available tools. I'll use mkdir to create the directory, then echo to create the file.

Let me start with mkdir projects.
thread_summary= Thread started. User wants to: 1) Create a 'projects' directory in the working dir. 2) Create a README.md file inside it with a hello world statement.
tool_call mkdir projects
output= 
step 1
dict_keys(['model', 'created_at', 'response', 'thinking', 'done', 'done_reason', 'context', 'total_duration', 'load_duration', 'prompt_eval_count', 'prompt_eval_duration', 'eval_count', 'eval_duration'])
thinking:  The user wants me to create a README.md file inside the 'projects' d

In [43]:
reply

{'tool_call': 'cat projects/README.md',
 'thread_summary': "Created 'projects' directory and created README.md with 'Hello World' content inside it. Now verifying the file exists and checking its contents."}